In [15]:
import torch
from torch import nn as nn
from torch import optim as optim
from google.colab import drive
drive.mount('/content/drive')
import pandas as pd
import re
from collections import Counter
from torch.utils.data import Dataset,DataLoader

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [16]:
df=pd.read_csv('/content/drive/MyDrive/Colab Notebooks/IMDB Dataset.csv')
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [17]:
df.duplicated().sum()

np.int64(418)

In [18]:
df.drop_duplicates(inplace=True)
df

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive
...,...,...
49995,I thought this movie did a down right good job...,positive
49996,"Bad plot, bad dialogue, bad acting, idiotic di...",negative
49997,I am a Catholic taught in parochial elementary...,negative
49998,I'm going to have to disagree with the previou...,negative


In [19]:
from sklearn.model_selection import train_test_split
X_train, X_val, y_train, y_val = train_test_split(
    df["review"],
    df["sentiment"],
    test_size=0.2,
    random_state=42,
    stratify=df["sentiment"]
)

In [20]:
def tokenize(text):
    text = text.lower()
    text = re.sub(r"<.*?>", "", text) 
    text = re.sub(r"[^a-zA-Z]", " ", text)
    return text.split()

In [21]:
def build_vocab(texts, max_vocab=30000):
    counter = Counter()

    for text in texts:
        counter.update(tokenize(text))

    vocab = {"<PAD>": 0, "<UNK>": 1}

    for idx, (word, _) in enumerate(counter.most_common(max_vocab - 2), start=2):
        vocab[word] = idx

    return vocab

vocab = build_vocab(X_train.tolist())

In [22]:
MAX_LEN = 200

def encode(text, vocab):
    tokens = tokenize(text)
    ids = [vocab.get(word, vocab["<UNK>"]) for word in tokens]

    if len(ids) < MAX_LEN:
        ids += [vocab["<PAD>"]] * (MAX_LEN - len(ids))
    else:
        ids = ids[:MAX_LEN]

    return ids

In [23]:
class ReviewDataset(Dataset):
    def __init__(self, texts, labels, vocab):
        self.texts = texts.tolist()
        self.labels = labels.tolist()
        self.vocab = vocab

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoded = encode(self.texts[idx], self.vocab)

        label = self.labels[idx]
        if label == "positive":
            label = 1
        elif label == "negative":
            label = 0

        return (
            torch.tensor(encoded, dtype=torch.long),
            torch.tensor(label, dtype=torch.float32)
        )

In [24]:
BATCH_SIZE = 64

train_dataset = ReviewDataset(X_train, y_train, vocab)
val_dataset   = ReviewDataset(X_val, y_val, vocab)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE)

In [29]:
class SimpleRNN(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.rnn = nn.RNN(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            batch_first=True
        )

        self.fc = nn.Linear(hidden_dim, 1)

    def forward(self, x):

        x = self.embedding(x)
        output, h_n = self.rnn(x)
        last_hidden = h_n[-1]
        logits = self.fc(last_hidden)
        return logits

In [34]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = SimpleRNN(     
    vocab_size=len(vocab),
    embed_dim=128,
    hidden_dim=128
).to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [37]:
EPOCHS = 5

for epoch in range(EPOCHS):

    model.train()
    total_loss = 0

    for inputs, labels in train_loader:
        inputs = inputs.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(inputs).squeeze(1)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()


    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs = inputs.to(device)
            labels = labels.to(device)

            outputs = model(inputs).squeeze(1)
            preds = torch.sigmoid(outputs) > 0.5

            correct += (preds.float() == labels).sum().item()
            total += labels.size(0)

    print(f"Epoch {epoch+1}")
    print(f"Train Loss: {total_loss/len(train_loader):.4f}")
    print(f"Val Accuracy: {correct/total:.4f}")


Epoch 1
Train Loss: 0.6671
Val Accuracy: 0.5122
Epoch 2
Train Loss: 0.6531
Val Accuracy: 0.5201
Epoch 3
Train Loss: 0.6310
Val Accuracy: 0.5260
Epoch 4
Train Loss: 0.6273
Val Accuracy: 0.5253
Epoch 5
Train Loss: 0.6166
Val Accuracy: 0.5264


In [38]:
correct += (preds == labels).sum().item()
total += labels.size(0)
accuracy = correct / total
accuracy

0.5266586490278613